# HW2 Data Processing

This notebook prepares the Homework 2 dataset for the current thesis task: predicting NISQ circuit reliability from pre-run circuit and hardware features. The goal is not to maximize model score here; it is to document the target, leakage boundary, grouped split logic, preprocessing policy, and saved train/validation/test datasets.


In [1]:
import importlib.util
import subprocess
import sys

_REQUIRED_IMPORTS = [
    ('pandas', 'pandas'),
    ('pyarrow', 'pyarrow'),
    ('sklearn', 'scikit-learn'),
]

_missing_pip_packages = []
for import_name, pip_name in _REQUIRED_IMPORTS:
    if importlib.util.find_spec(import_name) is None:
        _missing_pip_packages.append(pip_name)

if _missing_pip_packages:
    _cmd = [sys.executable, '-m', 'pip', 'install', *_missing_pip_packages]
    print('Installing missing notebook dependencies with:')
    print(' '.join(_cmd))
    subprocess.check_call(_cmd)
else:
    print('Notebook dependencies are available.')


Notebook dependencies are available.


## 1. Task Framing From HW1

Research problem: estimate how reliable a noisy simulated quantum circuit execution will be before using the observed output counts.

Analytical task: supervised regression.

Observation unit: one simulated circuit execution variant, either raw or transpiled.

Target variable: `reliability`, a bounded continuous value in `[0, 1]`.

Main metric for later benchmark comparison: validation MAE, supported by RMSE and R2.


In [2]:
import json
from pathlib import Path

import pandas as pd

FULL_SPLIT_DIR = Path('data/raw/thesis_production_125k/release/splits')
HW2_DATA_DIR = Path('data/hw2')

feature_policy = json.loads(
    (HW2_DATA_DIR / 'feature_policy.json').read_text(encoding='utf-8')
)
split_summary = json.loads(
    (HW2_DATA_DIR / 'split_summary.json').read_text(encoding='utf-8')
)
feature_manifest = json.loads(
    (FULL_SPLIT_DIR / 'feature_manifest.json').read_text(encoding='utf-8')
)

train = pd.read_parquet(HW2_DATA_DIR / 'train.parquet')
validation = pd.read_parquet(HW2_DATA_DIR / 'validation.parquet')
test = pd.read_parquet(HW2_DATA_DIR / 'test.parquet')

print('Saved HW2 split shapes:')
for name, frame in [('train', train), ('validation', validation), ('test', test)]:
    group_count = frame['base_circuit_id'].nunique()
    print(
        f'{name}: {frame.shape[0]:,} rows x {frame.shape[1]:,} columns; '
        f'groups={group_count:,}'
    )


Saved HW2 split shapes:
train: 20,000 rows x 75 columns; groups=10,000
validation: 2,500 rows x 75 columns; groups=1,250
test: 2,500 rows x 75 columns; groups=1,250


## 2. Data Description

This repository contains two datasets with different roles:

- Kaggle NISQ Fault Logs 100K: the earlier public dataset used for initial cleaning, feature engineering, and historical fault-type classification baselines with target `error_type`.
- Generated thesis dataset `thesis_production_125k_v1`: the current dataset produced by the separate dataset-generator workflow. It has 125,000 base circuits and 250,000 execution-variant rows because each base circuit has raw and transpiled variants.

HW2 uses the generated `thesis_production_125k_v1` dataset for reliability regression. The full local release contains an official grouped train/validation/test split. For GitHub submission, `data/hw2/` stores a reproducible grouped sample from that official split, because the full release files are too large to keep comfortably in the repository.

In [3]:
display(pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'saved_rows': [len(train), len(validation), len(test)],
    'saved_groups': [
        train['base_circuit_id'].nunique(),
        validation['base_circuit_id'].nunique(),
        test['base_circuit_id'].nunique(),
    ],
    'reliability_mean': [
        train['reliability'].mean(),
        validation['reliability'].mean(),
        test['reliability'].mean(),
    ],
    'reliability_min': [
        train['reliability'].min(),
        validation['reliability'].min(),
        test['reliability'].min(),
    ],
    'reliability_max': [
        train['reliability'].max(),
        validation['reliability'].max(),
        test['reliability'].max(),
    ],
}))

summary = pd.DataFrame({
    'dtype': train.dtypes.astype(str),
    'train_missing': train.isna().sum(),
    'train_unique_values': train.nunique(dropna=False),
})
summary.head(25)


,split,saved_rows,saved_groups,reliability_mean,reliability_min,reliability_max
0,train,20000,10000,0.789930,0.000977,1.0
1,validation,2500,1250,0.791551,0.010742,1.0
2,test,2500,1250,0.785319,0.016602,1.0


,dtype,train_missing,train_unique_values
dataset_id,object,0,1
profile_name,object,0,1
base_circuit_id,object,0,10000
circuit_id,object,0,20000
family,object,0,11
execution_mode,object,0,1
seed,int64,0,10000
compiler_variant,object,0,2
hardware_id,object,0,10000
hardware_snapshot_id,object,0,5


## 3. Missing Values And Duplicates

Some local hardware-mapping features are missing for raw circuit variants, because raw circuits do not have physical qubit mappings. These values are not filled globally in this notebook. The benchmark notebook handles missing values inside a train-only sklearn preprocessing pipeline.

The important duplicate-like structure is `base_circuit_id`: raw and transpiled variants can belong to the same base circuit. That is why the split must be grouped.


In [4]:
for name, frame in [('train', train), ('validation', validation), ('test', test)]:
    duplicate_circuit_ids = int(frame.duplicated(subset=['circuit_id']).sum())
    duplicate_base_ids = int(frame.duplicated(subset=['base_circuit_id']).sum())
    print(
        f'{name}: duplicate circuit_id rows={duplicate_circuit_ids:,}; '
        f'repeated base_circuit_id rows={duplicate_base_ids:,}'
    )

high_missing = summary[summary['train_missing'] > 0].sort_values('train_missing', ascending=False)
high_missing[['dtype', 'train_missing']].head(15)


train: duplicate circuit_id rows=0; repeated base_circuit_id rows=10,000
validation: duplicate circuit_id rows=0; repeated base_circuit_id rows=1,250
test: duplicate circuit_id rows=0; repeated base_circuit_id rows=1,250


,dtype,train_missing
noise_regime,object,20000
noise_dominant_channel,object,20000
mitigated_reliability,object,20000
mitigation_gain,object,20000
exact_match_probability,object,20000
local_readout_error_mean,float64,10000
local_two_qubit_error_max,float64,10000
local_two_qubit_error_mean,float64,10000
local_t2_mean,float64,10000
local_t1_mean,float64,10000


## 4. Leakage Audit

The prediction context is pre-run reliability estimation. The target is derived from noisy simulation outputs, but those post-run outputs must not be model inputs. The allowed inputs are circuit structure, compiler/hardware metadata, and calibration-like noise features available before execution.


In [5]:
print('Target:', feature_policy['target_column'])
print('Prediction context:', feature_policy['prediction_context'])
print()
print('Excluded leakage columns:')
for column, reason in feature_policy['excluded_leakage_columns'].items():
    print(f'- {column}: {reason}')
print()
print('Allowed input feature count:', len(feature_policy['allowed_feature_columns']))


Target: reliability
Prediction context: pre_run_reliability_regression

Excluded leakage columns:
- fidelity: alternative post-run target/outcome quality, not an input for reliability prediction
- error_rate: post-run target-like outcome column
- algorithmic_success_probability: post-run outcome metric
- exact_output_success_rate: post-run outcome metric
- counts_json: measured output counts from execution
- ideal_distribution_json: output distribution payload, not part of the pre-run scalar feature set
- exact_match_probability: post-run or mitigation-derived outcome field
- mitigated_reliability: post-run mitigation-derived outcome field
- mitigation_gain: post-run mitigation-derived outcome field
- noise_regime: post-generation label/diagnostic, not needed for pre-run benchmark inputs
- noise_dominant_channel: post-generation label/diagnostic, not needed for pre-run benchmark inputs

Allowed input feature count: 46


## 5. Split Design

Rows are grouped by `base_circuit_id` so raw/transpiled versions of the same underlying circuit cannot cross train, validation, and test. This matches the real prediction scenario: performance should be estimated on circuits whose base construction was not seen during fitting.

The test set is saved now but not used for HW2 model comparison.


In [6]:
train_groups = set(train['base_circuit_id'])
validation_groups = set(validation['base_circuit_id'])
test_groups = set(test['base_circuit_id'])
checks = {
    'train_validation_overlap': len(train_groups & validation_groups),
    'train_test_overlap': len(train_groups & test_groups),
    'validation_test_overlap': len(validation_groups & test_groups),
}
print(checks)
assert all(value == 0 for value in checks.values())

print(json.dumps(split_summary, indent=2)[:2500])


{'train_validation_overlap': 0, 'train_test_overlap': 0, 'validation_test_overlap': 0}
{
  "dataset_id": "thesis_production_125k_v1",
  "target_column": "reliability",
  "target_range": "[0, 1]",
  "task_type": "regression",
  "group_column": "base_circuit_id",
  "split_policy": "precomputed grouped split; base_circuit_id does not cross train/validation/test",
  "full_split_row_counts": {
    "train": 200000,
    "validation": 25000,
    "test": 25000
  },
  "hw2_saved_sample_row_counts": {
    "train": 20000,
    "validation": 2500,
    "test": 2500
  },
  "hw2_saved_sample_group_counts": {
    "train": 10000,
    "validation": 1250,
    "test": 1250
  },
  "test_set_policy": "saved now, held out from HW2 benchmark/model selection",
  "random_state_for_hw2_sampling": 42,
  "train_reliability_summary": {
    "mean": 0.7899298828125,
    "median": 0.845703125,
    "min": 0.0009765625,
    "max": 1.0
  },
  "validation_reliability_summary": {
    "mean": 0.79155078125,
    "median": 0.84

## 6. Saved Dataset Export

The HW2 datasets are already saved under `data/hw2/`. This cell rewrites them from the currently loaded frames so the notebook is executable from top to bottom and leaves the expected artifacts in place.


In [7]:
HW2_DATA_DIR.mkdir(parents=True, exist_ok=True)
train.to_parquet(HW2_DATA_DIR / 'train.parquet', index=False)
validation.to_parquet(HW2_DATA_DIR / 'validation.parquet', index=False)
test.to_parquet(HW2_DATA_DIR / 'test.parquet', index=False)
(HW2_DATA_DIR / 'feature_policy.json').write_text(
    json.dumps(feature_policy, indent=2), encoding='utf-8'
)
(HW2_DATA_DIR / 'split_summary.json').write_text(
    json.dumps(split_summary, indent=2), encoding='utf-8'
)

for path in sorted(HW2_DATA_DIR.glob('*')):
    if path.is_file():
        print(f'{path}: {path.stat().st_size:,} bytes')


data\hw2\feature_policy.json: 2,976 bytes
data\hw2\README.md: 1,553 bytes
data\hw2\split_summary.json: 1,278 bytes
data\hw2\test.parquet: 1,025,382 bytes
data\hw2\train.parquet: 7,681,724 bytes
data\hw2\validation.parquet: 987,852 bytes


## 7. Preprocessing Policy

Preprocessing that learns from data is intentionally deferred to `benchmark.ipynb`, where it is implemented as an sklearn pipeline fitted only on the training split. This avoids fitting imputers, encoders, or scalers on validation/test rows.

Methodological lesson: for this project, the critical HW2 decision is grouped splitting by `base_circuit_id` plus a clear pre-run feature boundary for reliability prediction.
